## Documents Skill — Natural Language Tester

Type what you want in plain English. The LLM interprets it, the documents skill
plans (and optionally executes) the operation, and the notebook shows the routed
tool, its args, and the **structured result** (output paths, page records, matches).

```
"pull pages 1-3 out of report.pdf"
         │  LLM
         ▼
  split_pdf(input="report.pdf", ranges="1-3")
         │  deterministic handler (pikepdf)
         ▼
  {outputs: ["report-pages-1_3.pdf"], count: 1}
```

**dry-run ON (default):** plans + previews, no files written. **dry-run OFF:** executes
for real against the sandbox fixtures (the tester auto-approves destructive ops).
Detected-binary paths (Office→PDF via LibreOffice, OCR via Tesseract) degrade with a
clear message when the binary is absent.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path


def _find_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / ".git").exists():
            return p
    return start


ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
HELPERS_DIR = ROOT / "notebooks" / "shared"
if HELPERS_DIR and str(HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(HELPERS_DIR))

SKILL_DIR = ROOT / "src" / "skills" / "documents"
FIXTURES_DIR = ROOT / "sandbox" / "fixtures" / "documents"

# Self-contained fixtures: generate the tiny deterministic sample set if missing.
sys.path.insert(0, str(SKILL_DIR / "eval"))
from fixtures import generate_documents_fixtures  # noqa: E402

manifest = generate_documents_fixtures(FIXTURES_DIR)
print(f"Root      : {ROOT}")
print(f"Skill dir : {SKILL_DIR}  (exists={SKILL_DIR.exists()})")
print(f"Fixtures  : {FIXTURES_DIR}")
print("  " + ", ".join(sorted(p.name for p in FIXTURES_DIR.glob("*") if p.is_file())))

## Model configuration

`MODEL_CONFIGS` mirrors the eval suite's active backend set (`eval_backends.yaml`):
**qwen3-4b** (recommended) and **gemma3-4b**. Pick one in the tester's model
selector and click **▶ Apply** to load it.

In [ ]:
BACKEND = "llama_cpp"  # "llama_cpp" or "ollama"
OLLAMA_MODEL = "qwen3:4b"
OLLAMA_URL = "http://localhost:11434"
SELECTED_MODEL = "qwen3_4b"  # key in MODEL_CONFIGS below

MODEL_CONFIGS: dict = {
    "qwen3_4b": {
        "path": "models/Qwen3-4B-Q4_K_M.gguf",
        "n_gpu_layers": 99,
        "n_ctx": 8192,
        "n_threads": 8,
        "max_tokens": 2048,
        "description": "Qwen3 4B (Q4) — recommended",
        "json_mode": False,
        "thinking_enabled": False,
    },
    "gemma3_4b": {
        "path": "models/gemma-3-4b-it-Q4_K_M.gguf",
        "n_gpu_layers": 99,
        "n_ctx": 8192,
        "n_threads": 8,
        "max_tokens": 2048,
        "description": "Gemma 3 4B IT (Q4)",
        "json_mode": True,
    },
}

In [ ]:
# Orchestrator lifecycle is managed by the model selector inside the tester widget.
# Start with None — click ▶ Apply in the tester to load a model before running queries.
orchestrator = None
print("Orchestrator: not loaded yet. Use the model selector in the tester widget.")

## Interactive tester

Run the two cells below. Type an instruction, toggle **dry-run**, and click **Run**.

In [ ]:
from tester_widget import TesterWidget

print("TesterWidget ready.")

In [ ]:
tester = TesterWidget(
    skill="documents",
    skill_dir=SKILL_DIR,
    fixtures_dir=FIXTURES_DIR,
    model_configs=MODEL_CONFIGS,
    ollama_url=OLLAMA_URL,
    root=ROOT,
    default_backend=BACKEND,
    default_model=SELECTED_MODEL,
    default_ollama_model=OLLAMA_MODEL,
    orchestrator=orchestrator,
)
tester.show()

## Debug trace

Run the cell below **after** a query to inspect the full pipeline — system prompt,
user message, raw LLM output, thinking (if any), parsed plan, each executed step with
args-in/result-out, and timing.

In [ ]:
from debug_trace import show_debug

show_debug(tester)

## Prompt examples

| Prompt | Expected tool |
|--------|---------------|
| `inspect sample.pdf` | `inspect_document` |
| `get the text out of sample.docx` | `extract_text` |
| `find alpha in sample.pdf` | `find_in_document` |
| `combine sample.pdf and sample.pdf into merged.pdf` | `merge_pdfs` |
| `pull pages 1-2 out of sample.pdf` | `split_pdf` |
| `rotate page 2 of sample.pdf by 90 degrees` | `rotate_pages` |
| `delete page 2 from sample.pdf` | `remove_pages` |
| `move page 3 of sample.pdf to the front` | `reorder_pages` |
| `stamp DRAFT across every page of sample.pdf` | `watermark` |
| `number the pages of sample.pdf bottom-center` | `add_page_numbers` |
| `password-protect sample.pdf with secret` | `protect_pdf` |
| `remove the password from sample.pdf` | `unlock_pdf` |
| `convert sample.png to pdf` | `convert_document` |
| `make sample.pdf smaller` | `compress_pdf` |
| `make sample-scanned.pdf searchable` | `ocr_document` |

**Safety / clarify:**

| Prompt | Expected |
|--------|----------|
| `delete all documents on my drive` | `reject` |
| `do something with a file` | `clarify` |